In [1]:
from science_jubilee.JubileeManager import JubileeManager
from science_jubilee.decks.LoadAll import *
from science_jubilee.tools.Loop import *
from science_jubilee.labware.Labware import *
from science_jubilee.decks.Deck import *
from science_jubilee.tools.Detection_erreur import Detection_erreur

In [2]:
# Connexion à la Jubilee
jm = JubileeManager(address="10.0.9.6", simulated=False)
ino = Loop
detect = Detection_erreur

2026-04-07 15:27:09 - [JubileeController]  - INFO - Initializing JubileeController (simulated=False, address=10.0.9.6)
2026-04-07 15:27:09 - [JubileeController]  - WARNING - Disconnecting this application from the network will halt connection to Jubilee.
2026-04-07 15:27:09 - [JubileeController]  - INFO - Connecting to Jubilee machine...
2026-04-07 15:27:10 - [JubileeController]  - INFO - Successfully connected and initialized Jubilee machine.
2026-04-07 15:27:10 - [JubileeManager]     - INFO - JubileeManager initialized (simulated=False, max_tools=4).


In [14]:
# Homing
jm.controller.home_all()

Is a tool currently mounted? [y/n]  n
Is the deck clear of any obstacles? [y/n]  y


2026-04-07 11:06:22 - [JubileeController]  - INFO - All axes homed successfully.


In [7]:
execute_plan('plan_jubilee', jm=jm)
#trace le plan

🚀 Exécution du plan Jubilee : plan_jubilee.txt
✅ Plan terminé avec succès sur Jubilee.


In [3]:
# Descendre le plateau pour positionner le deck
jm.controller.move_to(z=200)
jm.controller.move_to(x=150, y=150)

In [4]:
# Enregistrer le deck
deck = jm.load_deck("test1")
load_all(jm,"test1")

2026-04-07 15:27:22 - [Deck]               - INFO - Loading deck configuration from: /home/sworkyx/Documents/Polytech/Projet_indus/science-jubilee/src/science_jubilee/decks/deck_definition/test1.json
2026-04-07 15:27:22 - [Deck]               - INFO - Deck 'Experience1' loaded with 2 slots.
2026-04-07 15:27:22 - [JubileeManager]     - INFO - Deck 'Experience1' loaded from '/home/sworkyx/Documents/Polytech/Projet_indus/science-jubilee/src/science_jubilee/decks/deck_definition'.
2026-04-07 15:27:22 - [Tool]               - INFO - Tool 'Inoculator' (index 0) initialized.
2026-04-07 15:27:22 - [JubileeManager]     - INFO - Tool 'Inoculator' loaded at index 0.
2026-04-07 15:27:22 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, 36, 0).


In [5]:
#affiche les slots
jm.deck.list_slots()
jm.deck.get_all_slot_machine_coordinates()

{'0': (24.26, 84.45), '1': (192.55, 84.12)}

In [6]:
jm.status()

{'deck': 'Experience1',
 'tools': [{'index': 0, 'name': 'Inoculator'}],
 'active_tool_index': None,
 'active_tool_name': None,
 'tool_offsets': {0: (0, 36, 0)}}

In [7]:
# 1. On récupère les infos de ton outil par son nom
dict_outil = jm.get_tool_by_name("Inoculator")
idx = dict_outil['index']

# 2. On prépare l'outil physiquement
jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 68))

tool0 = jm.tools_list[idx]
tool0.__class__ = Loop
tool0._machine = jm.controller  # On le lie au contrôleur pour les mouvements
tool0.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    tool0.safe_z_movement = lambda: jm.controller.move_to(z=80)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{tool0.name}' prêt (Classe: {type(tool0).__name__})")

2026-04-07 15:27:30 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 0.
2026-04-07 15:27:37 - [JubileeController]  - INFO - pickup_tool_sequence completed for tool 0.
2026-04-07 15:27:37 - [JubileeManager]     - INFO - Tool at index 0 set as active tool.
2026-04-07 15:27:37 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, -31.5, 68).


✅ Correctif safe_z_movement appliqué.
✅ Outil 'Inoculator' prêt (Classe: Loop)


In [8]:
slot_1 = jm.deck.get_slot('1')
print(slot_1)
well_source = slot_1.labware.wells['A1'] ;
well_source.y = well_source.y - 31 + slot_1.coordinates[1] 
well_source.x = well_source.x + slot_1.coordinates[0] -7  #maj x
print(well_source)

Slot(slot_index='1', coordinates=(192.55, 84.12), shape='rectangle', width=127.76, length=85.57, diameter=None, has_labware=True, labware=reservoir: agilent_1_reservoir_290ml)
Well A1 at coordinates (249.43, 95.905, 4.82)


In [9]:
rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)

# On définit la SOURCE fixe (ici slot A1 du slot 
slot_0 = jm.deck.get_slot('0')
jm.controller.move_to(z=200)



print(f"🚀 Début de la distribution depuis {well_source.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6

        
        # On récupère l'objet Well de destination
        well_dest = slot_0.labware.wells[well_dest_name]

        well_dest.x = (well_dest.x + slot_0.coordinates[0] -7 )
        well_dest.y = (well_dest.y- 41 + slot_0.coordinates[1])#-41 ici car transferyt utilise move_to et pas effecteur to
        print(well_dest)

        print(f"Transfert de {well_source.name} vers {well_dest_name}...")

        try:
                # Appel du transfert (la sécurité safe_z est déjà injectée)
                jm.controller.move_to(z=200)        
                tool0.transfer(
                    source=well_source,      # FIXE
                    destination=well_dest,   # CHANGE à chaque itération
                    s=2000,
                    sweep_x=3,
                    sweep_y=3,
                    sweep_z=10,
                    sweep_speed=150,
                    up_speed=800,
                    randomize_pickup=True
                )
        except Exception as e:
                print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
                # Sécurité : on remonte immédiatement
                jm.controller.move_to(z=200)
                break
                
        jm.controller.move_to(z=90)
        jm.controller.move_to(x=well_dest.x, y=well_dest.y+36+20)
        jm.controller.dwell(1000)
        detect.capture_octopi_image("apres")
        jm.controller.dwell(1000)

        compteur = 0      
        while(detect.detect_lens() == False and compteur < 3 ):
            compteur= compteur + 1
            jm.controller.move_to(z=200)
            try:
                    # Appel du transfert (la sécurité safe_z est déjà injectée)
                tool0.transfer(
                source=well_source,      # FIXE
                destination=well_dest,   # CHANGE à chaque itération
                s=2000,
                sweep_x=3,
                sweep_y=3,
                sweep_z=10,
                sweep_speed=150,
                up_speed=800,
                randomize_pickup=True
                    )
            except Exception as e:
                print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
                # Sécurité : on remonte immédiatement
                jm.controller.move_to(z=200)
                break 
            jm.controller.move_to(z=90)
            jm.controller.move_to(x=well_dest.x, y=well_dest.y+36+20)
            jm.controller.dwell(5000)
            detect.capture_octopi_image("apres")
            jm.controller.dwell(1000)

  


print("✅ Distribution terminée sur toute la plaque !")

🚀 Début de la distribution depuis A1...
Well A1 at coordinates (32.260000000000005, 58.45, 2.5)
Transfert de A1 vers A1...
Connexion à OctoPi...
Image capturée et enregistrée : apres.jpg
✅ Lentille détectée, nombre de lentilles :  2
Well A2 at coordinates (32.260000000000005, 77.95, 2.5)
Transfert de A1 vers A2...
Connexion à OctoPi...
Image capturée et enregistrée : apres.jpg
✅ Lentille détectée, nombre de lentilles :  1
Well A3 at coordinates (32.260000000000005, 97.45, 2.5)
Transfert de A1 vers A3...
Connexion à OctoPi...
Image capturée et enregistrée : apres.jpg
✅ Lentille détectée, nombre de lentilles :  4
Well A4 at coordinates (32.260000000000005, 116.95, 2.5)
Transfert de A1 vers A4...
Connexion à OctoPi...
Image capturée et enregistrée : apres.jpg
✅ Lentille détectée, nombre de lentilles :  9
Well A5 at coordinates (32.260000000000005, 136.45, 2.5)
Transfert de A1 vers A5...
Connexion à OctoPi...
Image capturée et enregistrée : apres.jpg
✅ Lentille détectée, nombre de lentille

In [10]:
# Descendre le plateau pour positionner le deck
jm.controller.move_to(z=200)
jm.controller.move_to(x=150, y=150)